# PyTrees — Working with Nested Data in JAX

This notebook explores **pytrees**, JAX's way of handling nested data structures like dictionaries, tuples, and lists of arrays.

### Why does this matter?

In real machine learning, model parameters are never a single array. They look like this:

```python
params = {
    'layer1': {'weights': ..., 'bias': ...},
    'layer2': {'weights': ..., 'bias': ...},
}
```

JAX needs a way to:
- **Walk through** these nested structures
- **Apply functions** to every array inside them
- **Pass them** to `jit`, `grad`, and `vmap` seamlessly

That's exactly what **pytrees** do.

## What is a PyTree?

A **pytree** is any nested structure made of:
- **Containers** (nodes): `list`, `tuple`, `dict`
- **Leaves**: anything that isn't a container — typically arrays, numbers, `None`

Think of it like a file system:
```
params/              ← dict (container)
├── layer1/          ← dict (container)
│   ├── weights      ← array (leaf)
│   └── bias         ← array (leaf)
└── layer2/          ← dict (container)
    ├── weights      ← array (leaf)
    └── bias         ← array (leaf)
```

JAX can flatten this tree, apply operations to every leaf, and reconstruct it — all automatically.

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp

### ✅ Exercise 61 — Inspect a PyTree's Leaves

The most basic operation: extract all the **leaves** (actual data) from a nested structure.

Imagine you have a bag inside a bag inside a box. `tree_leaves` opens everything and gives you just the items inside.

In [2]:
# A simple pytree — a dict containing arrays
params = {
    'weights': jnp.array([1.0, 2.0, 3.0]),
    'bias': jnp.array([0.1]),
}

# Extract all leaves (the actual arrays)
leaves = jax.tree.leaves(params)
print(f"Number of leaves: {len(leaves)}")
for i, leaf in enumerate(leaves):
    print(f"  Leaf {i}: {leaf}")

Number of leaves: 2
  Leaf 0: [0.1]
  Leaf 1: [1. 2. 3.]


### ✅ Exercise 62 — Understand PyTree Structure

`tree_structure` shows you the **skeleton** of your pytree — the container layout without any data.

This is useful for debugging: you can check that two pytrees have the same shape before combining them.

In [3]:
# Nested pytree — like real model params
params = {
    'layer1': {
        'W': jnp.ones((3, 2)),
        'b': jnp.zeros(3),
    },
    'layer2': {
        'W': jnp.ones((1, 3)),
        'b': jnp.zeros(1),
    },
}

# See the skeleton
structure = jax.tree.structure(params)
print(f"Structure: {structure}")

# Count total leaves
leaves = jax.tree.leaves(params)
print(f"Total leaves: {len(leaves)}")
for leaf in leaves:
    print(f"  shape={leaf.shape}")

Structure: PyTreeDef({'layer1': {'W': *, 'b': *}, 'layer2': {'W': *, 'b': *}})
Total leaves: 4
  shape=(3, 2)
  shape=(3,)
  shape=(1, 3)
  shape=(1,)


### ✅ Exercise 63 — tree_map: Apply a Function to Every Leaf

`tree_map` is the workhorse of pytrees. It applies a function to **every leaf** in the tree and returns a new tree with the same structure.

Think of it like Python's built-in `map()`, but it works on nested structures:
```
map(f, [a, b, c])         → [f(a), f(b), f(c)]           # flat list
tree_map(f, nested_thing)  → same structure, f applied to every leaf
```

In [4]:
params = {
    'layer1': {
        'W': jnp.ones((3, 2)),
        'b': jnp.zeros(3),
    },
    'layer2': {
        'W': jnp.ones((1, 3)),
        'b': jnp.zeros(1),
    },
}

# Double every parameter in the tree
doubled = jax.tree.map(lambda x: x * 2, params)
print("layer1 W:", doubled['layer1']['W'])
print("layer2 b:", doubled['layer2']['b'])

layer1 W: [[2. 2.]
 [2. 2.]
 [2. 2.]]
layer2 b: [0.]


### ✅ Exercise 64 — tree_map with Multiple Trees

`tree_map` can take **multiple trees** as arguments, as long as they have the **same structure**. It zips them leaf-by-leaf.

This is exactly how gradient descent works:
```
new_params = params - learning_rate * gradients
```
Both `params` and `gradients` are pytrees with the same structure.

In [5]:
# Simulating a gradient descent step
params = {
    'W': jnp.array([1.0, 2.0, 3.0]),
    'b': jnp.array([0.5]),
}

# Pretend these are gradients (same structure as params)
grads = {
    'W': jnp.array([0.1, 0.2, 0.3]),
    'b': jnp.array([0.05]),
}

lr = 0.01

# SGD update: param = param - lr * grad
new_params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
print("Before:", params['W'])
print("After: ", new_params['W'])

Before: [1. 2. 3.]
After:  [0.999 1.998 2.997]


### ✅ Exercise 65 — Flatten and Unflatten

Sometimes you need to convert a pytree into a flat list of arrays and back. This is useful for:
- Serializing parameters
- Applying a function that expects flat arrays
- Debugging

`tree_flatten` gives you: (list of leaves, treedef)  
`tree_unflatten` puts them back together.

In [6]:
params = {
    'layer1': {'W': jnp.ones((2, 2)), 'b': jnp.zeros(2)},
    'layer2': {'W': jnp.ones((1, 2)), 'b': jnp.zeros(1)},
}

# Flatten: tree → (flat list, structure recipe)
flat_leaves, treedef = jax.tree.flatten(params)
print(f"Flat leaves ({len(flat_leaves)} arrays):")
for i, leaf in enumerate(flat_leaves):
    print(f"  [{i}] shape={leaf.shape}")

print(f"\nTreedef (recipe to rebuild): {treedef}")

# Unflatten: (flat list, structure recipe) → tree
rebuilt = jax.tree.unflatten(treedef, flat_leaves)
print(f"\nRebuilt layer1 W: {rebuilt['layer1']['W']}")

Flat leaves (4 arrays):
  [0] shape=(2, 2)
  [1] shape=(2,)
  [2] shape=(1, 2)
  [3] shape=(1,)

Treedef (recipe to rebuild): PyTreeDef({'layer1': {'W': *, 'b': *}, 'layer2': {'W': *, 'b': *}})

Rebuilt layer1 W: [[1. 1.]
 [1. 1.]]


### ✅ Exercise 66 — Count Total Parameters

A common utility: count how many numbers (parameters) are in your entire model. This uses `tree_map` to get the size of each leaf, then sums them up.

In [7]:
params = {
    'layer1': {'W': jnp.ones((128, 64)), 'b': jnp.zeros(128)},
    'layer2': {'W': jnp.ones((64, 128)), 'b': jnp.zeros(64)},
    'head':   {'W': jnp.ones((10, 64)),  'b': jnp.zeros(10)},
}

# Get size of each leaf, then sum
total = sum(jax.tree.leaves(jax.tree.map(lambda x: x.size, params)))
print(f"Total parameters: {total:,}")

Total parameters: 17,226


### ✅ Exercise 67 — Initialize Parameters with PyTrees

Real neural networks initialize their parameters randomly. Here we build a helper that creates a pytree of random weights given a list of layer sizes.

Notice how we split PRNG keys — one per parameter — just like we learned in notebook 30.

In [8]:
def init_mlp(key, layer_sizes):
    """Initialize an MLP with random weights."""
    params = []
    for i in range(len(layer_sizes) - 1):
        key, k1, k2 = jax.random.split(key, 3)
        W = jax.random.normal(k1, (layer_sizes[i+1], layer_sizes[i])) * 0.01
        b = jnp.zeros(layer_sizes[i+1])
        params.append({'W': W, 'b': b})
    return params

key = jax.random.PRNGKey(42)
params = init_mlp(key, [784, 128, 64, 10])  # MNIST-style network

print(f"Number of layers: {len(params)}")
for i, layer in enumerate(params):
    print(f"  Layer {i}: W={layer['W'].shape}, b={layer['b'].shape}")

total = sum(jax.tree.leaves(jax.tree.map(lambda x: x.size, params)))
print(f"Total parameters: {total:,}")

W0305 00:24:11.205511 10571595 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Number of layers: 3
  Layer 0: W=(128, 784), b=(128,)
  Layer 1: W=(64, 128), b=(64,)
  Layer 2: W=(10, 64), b=(10,)
Total parameters: 109,386


### ✅ Exercise 68 — grad Works on PyTrees Automatically

Here's the magic: `jax.grad` **already understands pytrees**. When you differentiate a function that takes a pytree as input, JAX returns gradients as a pytree with the **exact same structure**.

This means you don't need to manually match gradients to parameters — the structure does it for you.

In [9]:
# Simple 2-layer forward pass
def forward(params, x):
    for layer in params:
        x = jnp.tanh(layer['W'] @ x + layer['b'])
    return jnp.sum(x)

key = jax.random.PRNGKey(0)
params = init_mlp(key, [3, 4, 2])
x = jnp.array([1.0, 2.0, 3.0])

# grad returns a pytree with the SAME structure as params
grads = jax.grad(forward)(params, x)

print("Params structure:")
for i, layer in enumerate(params):
    print(f"  Layer {i}: W={layer['W'].shape}, b={layer['b'].shape}")

print("\nGrads structure (identical):")
for i, layer in enumerate(grads):
    print(f"  Layer {i}: W={layer['W'].shape}, b={layer['b'].shape}")

Params structure:
  Layer 0: W=(4, 3), b=(4,)
  Layer 1: W=(2, 4), b=(2,)

Grads structure (identical):
  Layer 0: W=(4, 3), b=(4,)
  Layer 1: W=(2, 4), b=(2,)


### ✅ Exercise 69 — Full Training Step with PyTrees

Combining everything: forward pass → compute loss → get gradients → update params. All using pytrees.

This is the **complete pattern** used in every JAX training loop.

In [10]:
def loss_fn(params, x, y_true):
    """MSE loss for a simple MLP."""
    y_pred = x
    for layer in params:
        y_pred = jnp.tanh(layer['W'] @ y_pred + layer['b'])
    return jnp.mean((y_pred - y_true) ** 2)

def train_step(params, x, y_true, lr=0.01):
    """One gradient descent step."""
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y_true)
    # tree_map applies the update to every matching leaf
    params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
    return params, loss

# Setup
key = jax.random.PRNGKey(0)
params = init_mlp(key, [3, 8, 2])
x = jnp.array([1.0, 0.5, -1.0])
y_true = jnp.array([1.0, 0.0])

# Train for a few steps
for step in range(5):
    params, loss = train_step(params, x, y_true)
    print(f"Step {step}: loss = {loss:.6f}")

Step 0: loss = 0.499461
Step 1: loss = 0.489487
Step 2: loss = 0.479714
Step 3: loss = 0.470143
Step 4: loss = 0.460772


### ✅ Exercise 70 — JIT-Compiled Training Step

The final piece: wrap the training step with `jit` for maximum speed. JAX's JIT understands pytrees natively, so this just works.

In [11]:
import time

# JIT the entire training step
@jax.jit
def fast_train_step(params, x, y_true):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y_true)
    params = jax.tree.map(lambda p, g: p - 0.01 * g, params, grads)
    return params, loss

# Setup
key = jax.random.PRNGKey(0)
params = init_mlp(key, [3, 64, 32, 2])
x = jnp.array([1.0, 0.5, -1.0])
y_true = jnp.array([1.0, 0.0])

# Warm up JIT
params_jit = params
params_jit, _ = fast_train_step(params_jit, x, y_true)

# Benchmark
def benchmark(fn, *args, runs=500):
    start = time.perf_counter()
    for _ in range(runs):
        result = fn(*args)
        jax.block_until_ready(result)
    return (time.perf_counter() - start) / runs

jit_time = benchmark(fast_train_step, params, x, y_true)
eager_time = benchmark(train_step, params, x, y_true)

print(f"JIT step  : {jit_time * 1e6:.0f} µs")
print(f"Eager step: {eager_time * 1e6:.0f} µs")
print(f"Speedup   : {eager_time / jit_time:.1f}x")

JIT step  : 16 µs
Eager step: 4510 µs
Speedup   : 281.1x


## Conclusion

### PyTrees are the Glue of JAX

Every JAX transformation (`jit`, `grad`, `vmap`) works with pytrees natively. This means:

| Operation | What happens with pytrees |
|---|---|
| `jax.grad(f)(params)` | Returns gradients as a pytree matching `params` |
| `jax.tree.map(f, tree)` | Applies `f` to every leaf, preserves structure |
| `jax.tree.leaves(tree)` | Extracts all arrays from any nesting |
| `jax.tree.flatten/unflatten` | Convert between flat list and nested tree |

### The pattern you'll use everywhere:
```python
# 1. Store params as pytree (dict/list of arrays)
# 2. Write forward/loss functions that take pytrees
# 3. Use grad() to get gradient pytrees
# 4. Use tree_map() to update params
# 5. Wrap with jit() for speed
```

This is the foundation of every JAX training loop.